In [1]:
!pip install gdown
!pip install py-vncorenlp
!pip install vncorenlp
!pip install rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 31.1 MB/s eta 0:00:00
  Created wheel for py-vncorenlp: filename=py_vncorenlp-0.1.4-py3-none-any.whl size=4304 sha256=7a614872cc162da803f55729a8249c2ef775b682508933c0786da7143cc27872
  Stored in directory: /root/.cache/pip/wheels/6d/2d/d6/158260bfd6820d144535857b80cc112bee5c3aa6d81b6dc049
Successfully built py-vncorenlp
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 49.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for vncorenlp: filename=vncorenlp-1.0.3-py3-none-any.whl size=2645933 sha256=3a394d66a077252636af25eb72011bbf715ab4c233aad31ff068d3ed6289d341
  Stored in directory: /root/.cache/pip/wheels/80/ad/d4/9e1a0939f63331a3898f2a951a368bbf0d69f7b027cae4d66b
Successfully built vncorenlp


In [2]:
import gdown
id_drive = '1OJDEhWCnmNOxNxoUakWtgbyOHQZHBcfz'
gdown.download(id=id_drive, output='news_summarization_vi_train.csv', quiet=False, fuzzy=True)

Downloading...
From (original): https://drive.google.com/uc?id=1OJDEhWCnmNOxNxoUakWtgbyOHQZHBcfz
From (redirected): https://drive.google.com/uc?id=1OJDEhWCnmNOxNxoUakWtgbyOHQZHBcfz&confirm=t&uuid=122f7576-1dd5-4f98-9561-d443557dfdae
To: /kaggle/working/news_summarization_vi_train.csv
100%|██████████| 181M/181M [00:03<00:00, 52.3MB/s]


'news_summarization_vi_train.csv'

# Config

In [3]:
import py_vncorenlp
py_vncorenlp.download_model(save_dir='./')

--2025-11-23 17:14:06--  https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.2.jar
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 27412703 (26M) [application/octet-stream]
Saving to: ‘VnCoreNLP-1.2.jar’

     0K .......... .......... .......... .......... ..........  0% 25.7M 1s
    50K .......... .......... .......... .......... ..........  0% 25.5M 1s
   100K .......... .......... .......... .......... ..........  0%  146M 1s
   150K .......... .......... .......... .......... ..........  0% 34.6M 1s
   200K .......... .......... .......... .......... ..........  0%  123M 1s
   250K .......... .......... .......... .......... ..........  1% 94.9M 1s
   300K .......... .......... .......... .......... ..........  1%  172M 1s
   350K ..

# Oreo preprocess

In [4]:
import pandas as pd
from pathlib import Path
from vncorenlp import VnCoreNLP
import nltk
from rouge_score import rouge_scorer
from tqdm import tqdm
import numpy as np
from transformers import AutoTokenizer
import re

rdr = VnCoreNLP("./VnCoreNLP-1.2.jar", annotators='wseg', max_heap_size='-Xmx2g')
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)
PHOBERT_MODEL = "bluenguyen/longformer-phobert-base-4096"
tokenizer = AutoTokenizer.from_pretrained(PHOBERT_MODEL)

tokenizer_config.json:   0%|          | 0.00/311 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

## Chuẩn bị Rouge

In [5]:
def eval_candidate(selected_sents, ref_summary):
    """
    Tính điểm quality cho 1 summary ứng viên
    Lấy trung bình Rouge-1 và Rouge-2
    """
    if not selected_sents:
        return 0.0

    cand = ' '.join(selected_sents)
    scores = scorer.score(ref_summary, cand)
    r1 = scores['rouge1'].fmeasure
    r2 = scores['rouge2'].fmeasure
    return (r1 + r2) / 2.0


## Tách câu

In [6]:
def split_sentences(text):
    sents = rdr.tokenize(text)
    sents = [' '.join(sent) for sent in sents if sent]
    sents = [re.sub(r'_', ' ', sent) for sent in sents]

    return sents

## Beam search sinh oracle summaries

In [7]:
def beam_search_oracles(sents, ref_summary, beam_size=64, max_steps=3, max_sents_doc=60):
    """
    Trả về list (idx_lít, rouge_score)
    idx_list: list index câu trong doc
    """

    if len(sents) > max_sents_doc:
        sents = sents[:max_sents_doc]

    beam = [([], 0.0, [])] # (indices, score, text_list)
    all_candidates = []

    for step in range(1, max_steps + 1):
        new_beam = []
        for idx_list, score, text_list in beam:
            used = set(idx_list)
            for i in range(len(sents)):
                if i in used:
                    continue
                new_idx = idx_list + [i]
                new_text_list = text_list + [sents[i]]
                new_score = eval_candidate(new_text_list, ref_summary)
                new_beam.append((new_idx, new_score, new_text_list))
                all_candidates.append((new_idx, new_score))

        new_beam.sort(key=lambda x: x[1], reverse=True)
        beam = new_beam[:beam_size]

    all_candidates.sort(key=lambda x: x[1], reverse=True)
    return all_candidates

## Oreo labels

In [8]:
def compute_oreo_labels(num_sents, oracles, top_t=16):
    """
    num_sents: số câu trong document
    orcales: list (idx_list, rouge_score)
    top_t: lấy bao nhiêu oracle tốt nhất để tính kỳ vọng
    """

    top_oracles = oracles[:top_t]
    raw_scores = np.zeros(num_sents, dtype=np.float32)

    if len(top_oracles) == 0:
        return raw_scores

    # uniform distribution trên top t oracles
    # l'_i += rouge_j nếu câu i nằm trong oracle j
    for idx_list, rouge in top_oracles:
        for i in idx_list:
            if 0 <= i < num_sents:
                raw_scores[i] += rouge

    max_val = raw_scores.max()
    min_val = raw_scores.min()

    if max_val == min_val:
        return raw_scores

    labels = (raw_scores - min_val) / (max_val - min_val)
    return labels

## Check length doc

In [9]:
def too_long_doc(sents, tokenizer, max_tokens=1024, min_tokens=64):
    """
    True nếu doc > max_tokens
    """
    if not sents:
        return False

    text = ' </s> '.join(sents)
    input_ids = tokenizer.encode(text, add_special_tokens=True)
    return len(input_ids) > max_tokens

## Raw → Extractive data

In [10]:
def process_dataset(input_path, output_path, beam_size=64, max_steps=3, top_t=16, len_df=10000, max_tokens=1024, min_tokens=64):
    """
    input_path: csv: content, summary
    output_path: csv: content, sentences: List[str], labels: List[float]
    """

    input_path = Path(input_path)
    output_path = Path(output_path)

    df = pd.read_csv(input_path)[:len_df]

    df = df.dropna(subset=['content', 'summary']).reset_index(drop=True)

    df = df[
        (df['content'].str.strip() != '') &
        (df['summary'].str.strip() != '') &
        (df['summary'].str.lower() != 'nan')
    ]


    out_data = []

    progress_bar = tqdm(df.iterrows(), total=len(df), desc="Processing dataset")

    for row in progress_bar:
        doc = row[1]['content']
        ref_sum = row[1]['summary']

        sents = split_sentences(doc)

        if len(sents) == 0:
            continue

        if too_long_doc(sents, tokenizer, max_tokens=max_tokens, min_tokens=min_tokens):
            continue

        # Bem -> oracle
        oracles = beam_search_oracles(sents, ref_sum, beam_size, max_steps)

        # Tính oreo labels
        labels = compute_oreo_labels(len(sents), oracles, top_t)

        out_item = {
            'content': doc,
            'sentences': sents,
            'labels': labels.tolist(),
            'summary': ref_sum
        }
        out_data.append(out_item)

        progress_bar.set_postfix({"Processed Samples": len(out_data)})



    out_df = pd.DataFrame(out_data)
    out_df.to_csv(output_path, index=False, encoding='utf-8')

In [11]:
process_dataset(
    input_path='./news_summarization_vi_train.csv',
    output_path='./oreo_news_summarization_vi_train.csv',
    beam_size=16,
    max_steps=2,
    top_t=4, len_df=30000
)

Processing dataset: 100%|██████████| 29509/29509 [8:16:07<00:00,  1.01s/it, Processed Samples=28251]
